## Phenotype verifier

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from explain.eval.tools.bio.phenotype import PhenotypeArgs, PhenotypeVerifier

/rxrx/data/user/lu.zhu/hooke-explain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

DIR = Path("/mnt/ps/home/CORP/lu.zhu/project/hooke-explain/data/curation_v1/results")
file = "structure-explain-results-v3-gemini2-5.csv"
# file = "structure-explain-results-v3-claude4.csv"
res = pd.read_csv(
    DIR / file,
    index_col=0,
)

In [3]:
# res = res[res["input_perturbation"].str.contains("TSC2") & res["explain"].str.contains("phenotype")]
res = res[res["explain"].str.contains("phenotype")]

res.head(10)
# all_reg_exp = [ rr for r in res.loc[res["explain"].str.contains("expression"), "explain"] for rr in r.split("\n") if 'regulates_expression' in rr]

,question,thinking,answer,explain,dag,raw_response,success,error,input_perturbation,input_report_text
index,,,,,,,,,,
0,How does the following perturbation influence ...,The task is to explain the mechanism of action...,"Bevacizumab, a recombinant humanized monoclona...","set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,# Comprehensive Mechanistic Report: Bevacizuma...
1,How does the following perturbation influence ...,1. **Context Understanding**: The initial con...,Nintedanib is a multi-tyrosine kinase inhibito...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n0"", ""n1"", relation=""causal"")\nedge(""n1""...",<think>\n1. **Context Understanding**: The in...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Nintedanib's Inhibitor...
2,How does the following perturbation influence ...,The user wants a structured explanation of Apa...,Apatinib is a small-molecule tyrosine kinase i...,"set_context(id=""n0"", cell_type=""vascular endot...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe user wants a structured explanati...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Apatinib in VEGF-Media...
3,How does the following perturbation influence ...,The task is to explain the mechanism of action...,Lenvatinib is a multi-targeted tyrosine kinase...,"set_context(id=""n0"", cell_type=""vascular endot...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Impact of Lenvatinib o...
4,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,"Motesanib, a multi-targeted tyrosine kinase in...","set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,# Motesanib: A Multi-Targeted Tyrosine Kinase ...
5,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,"Sunitinib, a multi-targeted receptor tyrosine ...","set_context(id=""n0"", cell_type=""endothelial ce...",NaN,<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Impact of Sunitinib on...
6,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,"Ponatinib, an ATP-competitive multi-targeted t...","set_context(id=""c0"", cell_type=""endothelial ce...",NaN,<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Impact of Ponatinib on...
7,How does the following perturbation influence ...,The task is to explain the mechanism of Regora...,Regorafenib is a multi-kinase inhibitor that p...,"set_context(id=""n0"", cell_type=""vascular endot...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Regorafenib inhibition...
8,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,RAF265 is a dual ATP-competitive kinase inhibi...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n3""...",<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Impact of RAF265 on VE...


In [4]:
index = 11
res_pert = res.loc[index, "input_perturbation"]
res_pert

"{'context': {'perturbation_type': 'soluble factor', 'description': 'Soluble factor addition of VEGF', 'cell_type': 'N/A', 'disease_model': 'Angiogenic factor/tumors'}, 'perturbation': {'type': 'chemical', 'smiles': 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4', 'name': 'Axitinib', 'target': 'VEGFR', 'moa_type': 'inhibitor'}}"

In [5]:
res_exp = res.loc[index, "explain"]

[rr for rr in res_exp.split("\n") if "perturbation" in rr]

['set_context(id="n0", cell_type="endothelial cells", disease_model="angiogenic tumors", prior_perturbation="VEGF stimulation")']

In [6]:
ph_statement = [rr.strip() for rr in res_exp.split("\n") if "phenotype" in rr]
ph_statement

['induces_phenotype(id="n8", source="Axitinib", phenotype="endothelial cell growth suppression", from_state="proliferating", to_state="G0/G1 arrest", via="cell cycle regulation")',
 'alleviates_phenotype(id="n9", actor="Axitinib", phenotype="endothelial cell migration", via="VEGFR inhibition")',
 'alleviates_phenotype(id="n10", actor="Axitinib", phenotype="angiogenic tube formation", via="VEGFR inhibition")',
 'induces_phenotype(id="n13", source="Axitinib", phenotype="pericyte proliferation inhibition", via="PDGFR inhibition")',
 'alleviates_phenotype(id="n14", actor="Axitinib", phenotype="pericyte migration", via="PDGFR inhibition")',
 'alleviates_phenotype(id="n15", actor="Axitinib", phenotype="vascular permeability", via="VEGFR inhibition")',
 'alleviates_phenotype(id="n16", actor="Axitinib", phenotype="tumor microvessel density", via="VEGFR inhibition")',
 'induces_phenotype(id="n17", source="Axitinib", phenotype="tumor cell apoptosis", via="VEGFR inhibition")',
 'alleviates_phenot

In [7]:
ph = ph_statement[0]
ph

'induces_phenotype(id="n8", source="Axitinib", phenotype="endothelial cell growth suppression", from_state="proliferating", to_state="G0/G1 arrest", via="cell cycle regulation")'

In [8]:
import re

matches = re.findall(r'(\w+)="([^"]*)"', ph)
args_dict = dict(matches)

In [9]:
args_dict

{'id': 'n8',
 'source': 'Axitinib',
 'phenotype': 'endothelial cell growth suppression',
 'from_state': 'proliferating',
 'to_state': 'G0/G1 arrest',
 'via': 'cell cycle regulation'}

In [10]:
pert_dict = eval(res_pert)
pert_dict

{'context': {'perturbation_type': 'soluble factor',
  'description': 'Soluble factor addition of VEGF',
  'cell_type': 'N/A',
  'disease_model': 'Angiogenic factor/tumors'},
 'perturbation': {'type': 'chemical',
  'smiles': 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4',
  'name': 'Axitinib',
  'target': 'VEGFR',
  'moa_type': 'inhibitor'}}

## Chemistry source

In [ ]:
# specify a llm model for text comparison tasks
from explain.llm import create_client

llm_client = create_client(provider="litellm", model="gemini-2.5-flash")

2025-08-20 23:30:24.173 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model gemini-2.5-flash
2025-08-20 23:30:24.174 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


### Example with alleviate phenotype

In [ ]:
# Define phenotype verfier tool object
pv = PhenotypeVerifier()

In [ ]:
# Define phenotype Arguments
mol = "Axitinib"
args = PhenotypeArgs(source_entity=mol, phenotype=["endothelial cell growth"], cell_type="HUVEC", direction="alleviate")

2025-08-20 23:30:24.249 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model gemini-2.5-flash
2025-08-20 23:30:24.249 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


Found CID for 'Axitinib': 6450551


2025-08-20 23:30:41.920 | INFO     | explain.eval.tools.bio.entity:_fetch_rxrx_id:221 - Fetching REC_ID for RITAVMQDGBJQJZ-FMIVXFBMSA-N
/rxrx/data/user/lu.zhu/hooke-explain/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Found CID for 'Axitinib': 6450551


2025-08-20 23:30:59.569 | INFO     | explain.eval.tools.bio.entity:_fetch_rxrx_id:221 - Fetching REC_ID for RITAVMQDGBJQJZ-FMIVXFBMSA-N
/rxrx/data/user/lu.zhu/hooke-explain/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# perform verification logic
pv._tool_logic(args, llm_client)
# reward and feedback

2025-08-20 23:31:04,556 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine


2025-08-20 23:31:04.583 | INFO     | explain.eval.tools.bio.entity:retrieve_identifiers:159 - NER model info: hunflair2
2025-08-20 23:31:25.334 | INFO     | explain.eval.tools.bio.entity:_ID_mapping:109 - ID mapping.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


(0.0,
 {'source_entity': CompoundEntity(name='Axitinib', ChEMBL='CHEMBL1289926', PubChem=6450551, REC_ID='REC-0000850', inchi_key='RITAVMQDGBJQJZ-FMIVXFBMSA-N', smiles='CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4', std_smiles=None, mesh_id=None, iupac_name='N-methyl-2-[[3-[(E)-2-pyridin-2-ylethenyl]-1H-indazol-6-yl]sulfanyl]benzamide', dose=None),
  'phenotype': {'requested': ['endothelial cell growth'],
   'results': array(['endothelial cell proliferation'], dtype=object)},
  'direction': {'requested': 'alleviate', 'results': False},
  'verification_status': 'NOT_VERIFIED'})

### Example with induced phenotype

In [16]:
mol = "Axitinib"
args = PhenotypeArgs(
    source_entity=mol,
    phenotype=["tumor cell apoptosis"],
    compounds=[mol],
    dose=None,
    cell_type="HUVEC",
    direction="induce",
)

2025-08-20 23:31:29.978 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model gemini-2.5-flash
2025-08-20 23:31:29.979 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


Found CID for 'Axitinib': 6450551


2025-08-20 23:31:45.411 | INFO     | explain.eval.tools.bio.entity:_fetch_rxrx_id:221 - Fetching REC_ID for RITAVMQDGBJQJZ-FMIVXFBMSA-N
/rxrx/data/user/lu.zhu/hooke-explain/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Found CID for 'Axitinib': 6450551


2025-08-20 23:32:02.132 | INFO     | explain.eval.tools.bio.entity:_fetch_rxrx_id:221 - Fetching REC_ID for RITAVMQDGBJQJZ-FMIVXFBMSA-N
/rxrx/data/user/lu.zhu/hooke-explain/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
pv._tool_logic(args, llm_client)
# reward and feedback

2025-08-20 23:32:05,544 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine


2025-08-20 23:32:05.561 | INFO     | explain.eval.tools.bio.entity:retrieve_identifiers:159 - NER model info: hunflair2
2025-08-20 23:32:26.368 | INFO     | explain.eval.tools.bio.entity:_ID_mapping:109 - ID mapping.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


(1.0,
 {'source_entity': CompoundEntity(name='Axitinib', ChEMBL='CHEMBL1289926', PubChem=6450551, REC_ID='REC-0000850', inchi_key='RITAVMQDGBJQJZ-FMIVXFBMSA-N', smiles='CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4', std_smiles=None, mesh_id=None, iupac_name='N-methyl-2-[[3-[(E)-2-pyridin-2-ylethenyl]-1H-indazol-6-yl]sulfanyl]benzamide', dose=None),
  'phenotype': {'requested': ['tumor cell apoptosis'],
   'results': array(['cellular apoptosis'], dtype=object)},
  'direction': {'requested': 'induce', 'results': True},
  'verification_status': 'VERIFIED'})